# Qwen3-Embedding-8B through the Nebius API

A smoke test of the hosted model before using it for the role probes: does the
key work, does the API serve `Qwen/Qwen3-Embedding-8B`, and do the billed tokens
match what we send? The same texts are embedded **without** and **with** an
instruction, in Qwen's `Instruct: {instruction}\nQuery:{text}` format
(`embedding_input`, the same function the local runs use).

Nebius Token Factory is OpenAI-compatible (`POST /v1/embeddings`), so a plain
HTTP request is enough; no SDK is needed.

The key is read from `[Nebius] NEBIUS_API_KEY` in `config.secrets.ini` (an
environment variable of the same name takes precedence). Never paste it into a cell.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from transformers import AutoTokenizer

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lab.text_to_features.embeddings import MODELS, embedding_input  # noqa: E402
from lab.text_to_features.role_probes import INSTRUCTIONS  # noqa: E402
from nba_ou.config.settings import SETTINGS  # noqa: E402

BASE_URL = "https://api.tokenfactory.nebius.com/v1"
MODEL = "Qwen/Qwen3-Embedding-8B"
# USD per million input tokens. Third-party trackers list $0.01; check the
# model's price in the Nebius console and update this before trusting the cost.
PRICE_PER_M_TOKENS = 0.01

# [Nebius] NEBIUS_API_KEY in config.secrets.ini, or the env var of the same name.
API_KEY = SETTINGS.nebius_api_key
session = requests.Session()
session.headers["Authorization"] = f"Bearer {API_KEY}"

## 1. Key and model

`GET /v1/models` checks the key; the model must be in the list.

In [2]:
response = session.get(f"{BASE_URL}/models", timeout=30)
response.raise_for_status()
model_ids = sorted(m["id"] for m in response.json()["data"])
print(len(model_ids), "models served")
print("Embedding models:", [m for m in model_ids if "embed" in m.lower()])
assert MODEL in model_ids, f"{MODEL} is not served"

30 models served
Embedding models: ['Qwen/Qwen3-Embedding-8B']


## 2. Embed with and without an instruction

One request per condition, every text in the same request. Each request
records the `usage` the API reports (what it bills) and the token count of
the same inputs with the local Qwen3 tokenizer (shared by the 0.6B, 4B and 8B
models; loaded from the cached 0.6B snapshot).

In [3]:
TEXTS = [
    "Luka Doncic scored 38 points and the Los Angeles Lakers beat the Minnesota "
    "Timberwolves 118-110. Rudy Gobert (left ankle) did not play.",
    "In this text, Target Team is the Boston Celtics (home team) and Opponent Team "
    "is the Miami Heat (visiting team).\n\nTarget Team beat Opponent Team 112-104 "
    "behind 31 points from Jayson Tatum.",
    "The Opponent Team erased a 15-point deficit in the fourth quarter and won in "
    "overtime.",
]
CONDITIONS = {"none": None, "target_view": INSTRUCTIONS["target_view"]}

tokenizer_id = "Qwen/Qwen3-Embedding-0.6B"
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_id,
    revision=MODELS[tokenizer_id][0],
    cache_dir=str(ROOT / "artifacts/huggingface"),
)


def embed(texts: list[str]) -> tuple[np.ndarray, dict, float]:
    started = time.perf_counter()
    response = session.post(
        f"{BASE_URL}/embeddings",
        json={"model": MODEL, "input": texts, "encoding_format": "float"},
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(f"{response.status_code}: {response.text[:500]}")
    body = response.json()
    rows = sorted(body["data"], key=lambda d: d["index"])
    vectors = np.array([row["embedding"] for row in rows], dtype=np.float32)
    return vectors, body.get("usage", {}), time.perf_counter() - started


vectors, rows = {}, []
for name, instruction in CONDITIONS.items():
    inputs = [embedding_input(text, instruction) for text in TEXTS]
    vectors[name], usage, seconds = embed(inputs)
    local_tokens = sum(len(ids) for ids in tokenizer(inputs)["input_ids"])
    billed = usage.get("prompt_tokens")
    rows.append(
        {
            "condition": name,
            "texts": len(inputs),
            "dimensions": vectors[name].shape[1],
            "mean_norm": float(np.linalg.norm(vectors[name], axis=1).mean()),
            "prompt_tokens": billed,
            "total_tokens": usage.get("total_tokens"),
            "local_tokens": local_tokens,
            "tokens_minus_local": None if billed is None else billed - local_tokens,
            "cost_usd": None if billed is None else billed * PRICE_PER_M_TOKENS / 1e6,
            "seconds": round(seconds, 2),
        }
    )

calls = pd.DataFrame(rows)
display(calls)
print("Raw usage of the last call:", json.dumps(usage))
print("Local tokenizer ends with:", repr(tokenizer.decode(tokenizer(inputs[0])["input_ids"][-1:])))

,condition,texts,dimensions,mean_norm,prompt_tokens,total_tokens,local_tokens,tokens_minus_local,cost_usd,seconds
0,none,3,4096,1.0,117,117,117,0,0.000001,1.72
1,target_view,3,4096,1.0,172,172,172,0,0.000002,1.72


Raw usage of the last call: {"prompt_tokens": 172, "total_tokens": 172, "completion_tokens": 0, "prompt_tokens_details": null}
Local tokenizer ends with: '<|endoftext|>'


`tokens_minus_local` should be 0, or one token per text if the server and the
local tokenizer disagree on the end-of-text token. A larger gap means the
server counts something else; compare with the Nebius console before scaling up.
`mean_norm` near 1 means the API returns L2-normalized vectors, like the local
recipe.

## 3. Does the instruction move the vectors?

Cosine between the two conditions for each text (same check as the local
4B notebook, now on 8B).

In [4]:
def normalize(x: np.ndarray) -> np.ndarray:
    return x / np.linalg.norm(x, axis=1, keepdims=True)


none, instructed = normalize(vectors["none"]), normalize(vectors["target_view"])
display(
    pd.DataFrame(
        {
            "text": [t[:60] + "..." for t in TEXTS],
            "cosine_none_vs_instructed": (none * instructed).sum(axis=1).round(4),
        }
    )
)
print("Pairwise cosine without instruction:\n", (none @ none.T).round(3))
print("Pairwise cosine with instruction:\n", (instructed @ instructed.T).round(3))

,text,cosine_none_vs_instructed
0,Luka Doncic scored 38 points and the Los Angel...,0.8089
1,"In this text, Target Team is the Boston Celtic...",0.8372
2,The Opponent Team erased a 15-point deficit in...,0.7818


Pairwise cosine without instruction:
 [[1.    0.558 0.475]
 [0.558 1.    0.575]
 [0.475 0.575 1.   ]]
Pairwise cosine with instruction:
 [[1.    0.531 0.564]
 [0.531 1.    0.558]
 [0.564 0.558 1.   ]]


## 4. Billing check

Total of this notebook's calls, to compare with the usage shown in the Nebius
console (Billing / Usage) once it updates, which can take a while. At ~$0.01 per
million tokens a run of this notebook costs a fraction of a cent, so the
console may round it to $0.00; the token count is the number to compare.

In [5]:
print(f"Requests: {len(calls)}")
print(f"Billed prompt tokens: {calls['prompt_tokens'].sum()}")
print(f"Estimated cost: ${calls['cost_usd'].sum():.8f} at ${PRICE_PER_M_TOKENS}/M tokens")

Requests: 2
Billed prompt tokens: 289
Estimated cost: $0.00000289 at $0.01/M tokens
